# Notebook 09 — Development results and accepted shortlist

## Read this first

**The review is complete. The final benchmark has not run.** The five decisions kept the existing ten-case reference tags and clarified the rubric. Accepting the shortlist selected models; it did not change their predictions or scores.

This notebook reuses saved predictions. It does not train models or make provider calls. The new twelve-case review is scored against those saved answers where available.

| Results shown | Demonstrations | What the numbers mean |
|---|---:|---|
| Search/newer-model experiment | 10 | Previous GPT-OSS-120B and Qwen results; unchanged after the five decisions |
| Earlier model comparison | 28 | Historical development comparison used to choose model families |
| Completed additional review | 12 | Production regex and TF-IDF have saved predictions for every case |
| Shared hosted-model subset | 5 of those 12 | All four candidates can be compared on these same five cases; seven lack saved hosted-check predictions |
| Reserved final benchmark | 123 event groups | Not run; there are no final scores yet |

Compare models **within the same sample**, never by comparing the twelve-case table with the five-case table. None of these development results measures the new rubric-v3 Qwen prompt.

**Review status:** The shortlist and five decisions are accepted, and all twelve reviews are recorded. No repeat labeling is requested here. The remaining DR-25 work is to prepare consistent final reference labels and validate the updated Qwen prompt before final execution.

Run all cells with the private reports available locally. Fingerprints check the exact saved inputs.


**Final evaluation is now complete:** This notebook preserves the development-selection record. See [Notebook 10](10_final_benchmark_and_recommendation.ipynb) and the [final report](../docs/plan/final-evaluation.md) for the approved 121-group benchmark and recommendation. Earlier pending-final statements below describe preparation at that time.


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Open this notebook inside the DemoLogic repository")
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "src"))
from notebooks.tools import percent, show_table, verify_sha256
from demo_logic.evaluation import multilabel_metrics
from demo_logic.taxonomy import topic_tags

folder = root / "reports/final-preparation/2026-10-01"
audit = json.loads((folder / "development-audit-v2.json").read_text())
contract = json.loads((folder / "candidate-contract-v3.json").read_text())
for relative, expected in {
    **audit["sources"],
    **contract["input_fingerprints"],
}.items():
    verify_sha256(root / relative, expected)
verify_sha256(root / "docs/tagging-rubric.md", audit["rubric_sha256"])
records = audit["records"]
assert len(records) == len({r["external_id"] for r in records}) == 10
print("Saved inputs verified. Ten development cases; no new inference.")

# Match copied audit answers back to the original reports by event ID.
source_predictions = {}
source_references = {}
for relative in audit["sources"]:
    source = json.loads((root / relative).read_text())
    if "answers" in source:
        prefix = "oss" if "single-model-v4" in relative else "qwen"
        for setting, answers in source["answers"].items():
            assert len(answers) == len({a["external_id"] for a in answers}) == 10
            source_predictions[f"{prefix}:{setting}"] = {
                answer["external_id"]: answer["predicted"] for answer in answers
            }
    else:
        source_references.update({row["external_id"]: row for row in source["records"]})
for row in records:
    assert set(row["reference_tags"]) == set(
        source_references[row["external_id"]]["tags"]
    )
    for setting, predicted in row["predictions"].items():
        assert predicted == source_predictions[setting][row["external_id"]]


## 1. Replay the ten-case comparison

The four configurations use the same ten accepted references and a strict two-tag policy. These context-sensitive cases are a small development diagnostic, not a representative accuracy estimate. The completed review retained their existing labels, with explicit limitations for references that rely on unverified background. Their scores therefore remain unchanged.

In [ ]:
tags = topic_tags(
    json.loads((root / "data/manifest.json").read_text())["taxonomy_version"]
)
reference = [row["reference_tags"] for row in records]
configurations = list(records[0]["predictions"])
assert all(set(row["predictions"]) == set(configurations) for row in records)
metrics = {
    name: multilabel_metrics(
        reference, [row["predictions"][name] for row in records], tags
    )
    for name in configurations
}
show_table(
    ["Configuration", "Precision", "Recall", "F1", "Coverage"],
    [
        [
            name,
            *[
                percent(result[key])
                for key in ("micro_precision", "micro_recall", "micro_f1", "coverage")
            ],
        ]
        for name, result in metrics.items()
    ],
)
fig, ax = plt.subplots(figsize=(10, 4))
labels = [
    name.replace("qwen:", "").replace("oss:", "GPT-OSS-120B ") for name in metrics
]
ax.barh(labels, [100 * m["micro_f1"] for m in metrics.values()])
ax.set(
    xlim=(0, 100),
    xlabel="Micro F1 (%)",
    title="Same ten development cases — accepted references",
)
fig.tight_layout()
display(fig)
plt.close(fig)


**Finding:** Qwen3.6 gives the highest F1 in this comparison. Qwen3.8 improves precision but misses more reference tags and abstains twice. Search does not improve GPT-OSS-120B's F1 on these cases.

## 2. Completed reference review

Four cases had full agreement and six had disagreements. Five requested decisions are now recorded. The review clarifies antiracism, homelessness support and the meaning of work under basic income. These decisions create rubric-v3. Two retained answers still depend on background or source matching that has not been verified; they are documented limitations, not confirmed model mistakes.

In [ ]:
show_table(
    ["Case", "Accepted tags", "Recorded decision", "Remaining evidence limitation"],
    [
        [
            row["external_id"],
            "; ".join(row["reference_tags"]),
            row["completed_decision"],
            row.get("reference_limitation") or "None recorded",
        ]
        for row in records
        if row["review_status"] == "reviewed"
    ],
)
print(
    "Five decisions recorded. Reference tag sets retained; ten-case metrics are unchanged."
)


## 3. Accepted shortlist

The accepted comparison includes production regex, word + character TF-IDF with logistic regression, and Qwen3.6-35B-A3B-FP8. Together they cover rules, a trained classifier and an LLM.

The table replays eight candidates on 28 reviewed development cases. Its older LLM prompts differ from the ten-case rubric-v2 prompt above and from the planned rubric-v3 prompt. These results support selecting model families; they do not measure the exact final configuration. GPT-OSS-20B remains a reserve.

In [ ]:
comparison = json.loads(
    (
        root
        / "reports/development-review/2026-10-01-random10/random10-approved-comparison-v1.json"
    ).read_text()
)
show_table(
    ["Saved candidate", "Precision", "Recall", "F1", "Coverage"],
    [
        [
            result["model"],
            *[
                percent(result["reviewed"][key])
                for key in ("micro_precision", "micro_recall", "micro_f1", "coverage")
            ],
        ]
        for result in comparison["batches"]["combined"]["models"].values()
    ],
)
show_table(
    ["Accepted candidate", "Configuration status"],
    [
        [
            "Production regex",
            "Preserved rules; positive matches; include exact ties at second rank",
        ],
        [
            "Word + character TF-IDF + logistic regression",
            "Saved training artifact; cutoff 0.30; include exact ties",
        ],
        [
            "Qwen/Qwen3.6-35B-A3B-FP8 (Hetzner)",
            "Saved results use rubric-v2. Rubric-v3 prompt still needs development validation; title-only, temperature 0, at most two tags.",
        ],
    ],
)


**Output policy:** Report a matched strict-two comparison for all candidates. Also show a separate exact-tie result for scored models. Keep the saved classifier cutoff at 0.30; no thresholds are selected using final answers.

## 4. Final preparation checks

The reserved evaluation contains 123 fixed event-group representatives from 149 reviewed occurrences. This cell verifies IDs and file versions without opening final reference answers. The completed twelve-case review is development evidence; its accepted tags and remaining limitations are recorded separately.

In [ ]:
selection = json.loads(
    (root / "data/benchmark/event-type-representatives-v1.json").read_text()
)["records"]
final = [row for row in selection if row["split"] == "final_test"]
validation_ids = {
    row["external_id"] for row in selection if row["split"] == "validation"
}
extra = json.loads((folder / "rubric-check-12-manifest.json").read_text())
extra_ids = {row["external_id"] for row in extra["records"]}
assert (
    len(final)
    == len({row["group_id"] for row in final})
    == contract["final_group_count"]
    == 123
)
assert len(extra_ids) == len(extra["records"]) == 12
assert extra_ids <= validation_ids
assert not extra_ids & {row["external_id"] for row in final}
verify_sha256(
    root / "data/benchmark/final-review-queue.csv",
    contract["current_final_review_file_sha256"],
)
print(
    "Preparation checks passed: 123 final groups and 12 separate development examples."
)
show_table(["Before final execution"], [[gate] for gate in contract["readiness_gates"]])
print("Final inference has NOT been run. Final reference answers were NOT loaded.")


In [ ]:
review = json.loads((folder / "rubric-check-12-reference-v2.json").read_text())
show_table(
    ["Review fact", "Value"],
    [
        ["Completed examples", len(review["records"])],
        ["Valid catalog tags", sum(row["score_eligible"] for row in review["records"])],
        [
            "Remaining unknown tags",
            sum(bool(row["unknown_tags"]) for row in review["records"]),
        ],
        ["Final-test examples", review["final_test_records_used"]],
    ],
)
show_table(
    ["Case", "Reviewed tags", "Recorded limitation"],
    [
        [
            row["external_id"],
            "; ".join(row["tags"]),
            "; ".join(
                row["limitations"]
                + (
                    ["Unknown tag: " + ", ".join(row["unknown_tags"])]
                    if row["unknown_tags"]
                    else []
                )
            ),
        ]
        for row in review["records"]
        if row["limitations"] or row["unknown_tags"]
    ],
)


## 5. Results on the twelve completed reviews

Compare rules and the trained classifier on all twelve reviewed development examples. Then compare all four saved candidates on the subset with hosted-check predictions. The subset size is shown explicitly; results from different sample sizes are not a direct model ranking. Older hosted prompts and retained rubric exceptions limit the interpretation.


In [ ]:
# Reuse existing answers only; do not infer missing predictions or call providers.
baseline_path = "reports/baselines/20260929T093729499041Z.json"
classical_path = "reports/classical/20260929T100051642372Z.json"
hosted_path = "reports/hosted/20260930T150128230609Z.json"
baseline = json.loads((root / baseline_path).read_text())
classical = json.loads((root / classical_path).read_text())
hosted = json.loads((root / hosted_path).read_text())
from notebooks.tools import file_sha256

reviewed = {row["external_id"]: row for row in review["records"]}
predictions = {
    "Production regex": {
        row["external_id"]: row["predicted"]
        for run in baseline["runs"]
        if run["candidate"] == "keywords" and run["policy"] == "strict_two"
        for row in run["predictions"]
    },
    "Word + character logistic regression": {
        row["external_id"]: row["predicted"]
        for run in classical["policies"]
        if run["model"] == "word_plus_char" and run["policy"] == "strict_two"
        for row in run["predictions"]
    },
}
for provider, name in [
    ("groq", "GPT-OSS-20B (older prompt)"),
    ("hetzner", "Qwen3.6 (older prompt)"),
]:
    predictions[name] = {
        row["external_id"]: row["predicted"] for row in hosted["check_runs"][provider]
    }
assert all(set(reviewed) <= set(predictions[name]) for name in list(predictions)[:2])
common_ids = sorted(set(reviewed).intersection(*(set(p) for p in predictions.values())))
assert common_ids
results = {}
for scope, ids, names in [
    ("All twelve reviews", sorted(reviewed), list(predictions)[:2]),
    ("Shared hosted-check subset", common_ids, list(predictions)),
]:
    scored = {
        name: multilabel_metrics(
            [reviewed[i]["tags"] for i in ids],
            [predictions[name][i] for i in ids],
            tags,
        )
        for name in names
    }
    results[scope] = {"ids": ids, "metrics": scored}
    print(f"{scope}: {len(ids)} development examples")
    show_table(
        ["Saved candidate", "Precision", "Recall", "F1", "Coverage"],
        [
            [
                name,
                *[
                    percent(result[key])
                    for key in (
                        "micro_precision",
                        "micro_recall",
                        "micro_f1",
                        "coverage",
                    )
                ],
            ]
            for name, result in scored.items()
        ],
    )
# Keep a private reproducible report. Only identical reruns may reuse its path.
from demo_logic.notebook_support import save_private_json

report = {
    "version": "completed-review-comparison-v1",
    "reference_sha256": file_sha256(folder / "rubric-check-12-reference-v2.json"),
    "rubric_version": "rubric-v3",
    "sources": {
        p: file_sha256(root / p) for p in [baseline_path, classical_path, hosted_path]
    },
    "results": results,
    "new_api_calls": 0,
    "final_test_records_used": 0,
    "limitations": "Targeted development selection; retained reviewer exceptions. Hosted predictions use older prompts and cover only the shared subset. No missing responses were imputed.",
}
save_private_json(folder / "completed-review-comparison-v1.json", report)


**Finding on all twelve reviews:** Production regex reaches 73.3% precision, 55.0% recall and 91.7% coverage. Word + character logistic regression reaches 80.0% precision, but recall is 40.0% and coverage is 58.3%. Meeting the precision target alone does not meet the overall requirements.

**Shared hosted subset:** Only five of these twelve examples have saved hosted-check answers. GPT-OSS-20B and Qwen3.6 both reach 92.3% F1 on those five. That is useful local evidence, but five targeted examples cannot establish an overall winner. It does not measure the newer rubric-v3 prompt or Qwen with search.


## Conclusion and next step

**Completed:** candidate selection, five development decisions, and twelve additional reviews. The `building` entries were confirmed as `housing`. The reference limitations are documented; no further answers are requested for those completed tasks.

**Current results:** The original ten-case metrics are unchanged. On the twelve new reviews, regex achieves 62.9% F1 and TF-IDF 53.3%. On the five-case shared subset, the older GPT-OSS-20B and Qwen3.6 prompts both achieve 92.3% F1. These describe different samples and cannot be combined into one final score or used as evidence for the untested rubric-v3 prompt.

**Next — DR-25:** Prepare the affected final-reference review and validate the updated Qwen prompt on development data, then implement and run Notebook 10 on the 123 fixed final groups. Historical training labels and documented reviewer exceptions remain limitations.

Search for Qwen remains an optional separate experiment, not a condition for completing this notebook. See `docs/plan/final-evaluation.md` for the final-run sequence.


**Subsequent DR-47 result:** Notebook 08a tested the revised rubric-v3 Qwen prompt on ten development cases. Title-only F1 was 81.5%, versus 66.7% with retrieved titles/URLs. Keep the title-only shortlist. The prepared contract above is the earlier snapshot; pin the validated DR-47 payload before final execution.
